In [ ]:
pip install powerlaw networkx numpy scipy pandas tqdm scipy

In [ ]:
import sys
import logging
from pathlib import Path

# sys.path manipulation is required because ysocial_validator is not installed as a package
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# scripts/ must be on sys.path for dynamic imports of numbered scripts (e.g., importlib.import_module("01_..."))
scripts_path = str(Path("../scripts").resolve())
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s")

from ysocial_validator.ingestion import YSocialGraphBuilder

db_path = Path("../data/00_raw/benchmark_runs/run01.sqlite").resolve()

G = YSocialGraphBuilder(str(db_path)).load_follower_graph()

In [ ]:
import pandas as pd
from ysocial_validator.topometrics import YSocialTopometrics

report = YSocialTopometrics(G).generate_full_report()

pd.Series(report).rename("value").to_frame()

In [ ]:
from importlib import import_module

stage_a_validator = import_module("01_stage_a_validator")
StageAValidator = stage_a_validator.StageAValidator

data_dir = Path("../data/00_raw/benchmark_runs").resolve()

validator = StageAValidator(str(data_dir))
raw_df = validator.process_runs()

output_csv = Path("../data/01_processed/stage_a_raw.csv").resolve()
validator.save_raw_results(str(output_csv))

report = validator.full_stability_report()
print("\n=== Topological Stability (Stage A) ===\n")
print(report["stability"])

In [ ]:
from importlib import import_module

stage_a_viz = import_module("02_stage_a_visualization")
StageAVisualizer = stage_a_viz.StageAVisualizer

viz = StageAVisualizer("../data/01_processed/stage_a_raw.csv")
viz.plot_stability_distributions("../data/01_processed/stage_a_stability.png")

# Optional: print text summary to stdout
print(viz.plot_summary_statistics())

In [ ]:
from importlib import import_module

stage_b_analyzer_mod = import_module("03_stage_b_analyzer")
StageBAnalyzer = stage_b_analyzer_mod.StageBAnalyzer

analyzer = StageBAnalyzer("../data/00_raw/sensitivity_runs/")
raw_df = analyzer.process_all_runs()
analyzer.save_raw_results("../data/01_processed/stage_b_raw.csv")
analyzer.save_aggregated_results("../data/01_processed/stage_b_aggregated.csv")

report = analyzer.full_sensitivity_report()
print(report["aggregated"])  # 11 rows, one per condition

In [ ]:
from importlib import import_module

stage_b_viz = import_module("04_stage_b_visualization")
StageBVisualizer = stage_b_viz.StageBVisualizer

viz = StageBVisualizer("../data/01_processed/stage_b_raw.csv")
viz.plot_modularity_comparison("../data/01_processed/stage_b_modularity_comparison.png")

# Optional: tabular modularity summary by condition
print(viz.get_summary_statistics())

In [ ]:
from importlib import import_module

stage_b_hyp = import_module("05_stage_b_hypothesis")
StageBHypothesisTesting = stage_b_hyp.StageBHypothesisTesting

tester = StageBHypothesisTesting("../data/01_processed/stage_b_raw.csv")
results_df = tester.run_tests()

tester.print_results(verbose=True)

tester.save_results("../data/01_processed/stage_b_pvalues.csv")

# Conditions with p < 0.05
sig_conditions = tester.get_significant_conditions(alpha=0.05)
print(f"Significant conditions (p<0.05): {sig_conditions}")